# 7  Better inputs, pipelines and AutoML

**Goal:** derive better inputs from dates and IDs, put preparation and model into one pipeline, and let an AutoML tool try many models.

**Data:** the late applications, without the leaking input.

**The data used here:**
- `late_applications.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/late_applications.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/late_applications.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

## Step 1  Derive better inputs

In [ ]:
#@title Step 1  Raw dates and IDs
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
df = pd.read_csv(DATA + "late_applications.csv", parse_dates=["documents_uploaded_at", "submitted_at"]).sort_values("submitted_at")
display(df[["application_id", "applicant_id", "documents_uploaded_at", "submitted_at", "late"]].head(5))
box("What you see", "A date or an ID tells a model little by itself. The people who run the service know what drives delays: "
    "how long the documents took, the day of the week, and whether the applicant has applied before.", "info")

In [ ]:
#@title Step 1b  Which derived input do you expect to matter most?  (choose, then click &#9654;)
guess = "Choose..." #@param ["Choose...", "Days between uploading documents and submitting", "Day of the week of submission", "Earlier applications by the same applicant"]
df["upload_gap_days"] = (df["submitted_at"] - df["documents_uploaded_at"]).dt.total_seconds() / 86400
df["weekday"] = df["submitted_at"].dt.day_name()
df["earlier_applications"] = df.groupby("applicant_id").cumcount()
BASE_CAT = ["service", "channel", "documents_complete", "medical_test_required", "security_check"]; BASE_NUM = ["number_of_documents", "queue_at_submission"]
y = df["late"]
def cv_acc(cat, num):
    X = pd.get_dummies(df[cat + num], dtype=int)
    return cross_val_score(RandomForestClassifier(n_estimators=150, random_state=42), X, y, cv=5).mean()
base = cv_acc(BASE_CAT, BASE_NUM)
gain = {"Days between uploading documents and submitting": cv_acc(BASE_CAT, BASE_NUM + ["upload_gap_days"]) - base,
        "Day of the week of submission": cv_acc(BASE_CAT + ["weekday"], BASE_NUM) - base,
        "Earlier applications by the same applicant": cv_acc(BASE_CAT, BASE_NUM + ["earlier_applications"]) - base}
bars({k: v * 100 for k, v in gain.items()}, "Gain in accuracy when the input is added (percentage points)", highlight=max(gain, key=gain.get), fmt="+{:.1f}")
if guess in gain: box("Your guess" + (" was right" if guess == max(gain, key=gain.get) else ""), f"The largest gain comes from: <b>{max(gain, key=gain.get)}</b>.", "ok" if guess == max(gain, key=gain.get) else "try")
else: box("Make your guess", "Pick an option on the right and click &#9654; again.", "try")

## Step 2  One pipeline

In [ ]:
#@title Step 2  Build the pipeline
CAT = BASE_CAT + ["weekday"]; NUM = BASE_NUM + ["upload_gap_days", "earlier_applications"]
prep = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), CAT),
                          ("numbers", Pipeline([("fill gaps", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUM)])
pipe = Pipeline([("preparation", prep), ("model", RandomForestClassifier(n_estimators=200, random_state=42))])
X_train, X_test, y_train, y_test = train_test_split(df[CAT + NUM], y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
display(pipe)
big((f"{pipe.score(X_test, y_test):.1%}", "right on the test data"))
box("What you see", "Preparation and model as one sequence (click the boxes to open them). Training, testing and production all run exactly the same steps.", "info")

In [ ]:
#@title Step 2b  A new application arrives  (pick an application, then click &#9654;)
application = 1 #@param {type:"slider", min:1, max:20, step:1}
new = X_test.iloc[[application - 1]]
display(new.T.rename(columns=lambda c: "new application"))
p = pipe.predict_proba(new)[0, 1]
box(f"Probability of being late: {p:.0%}", "The raw application went through the same preparation as the training data, then the model.", "see" if p >= .5 else "ok")

## Step 3  AutoML

In [ ]:
#@title Step 3  Let AutoML try many models  (click &#9654; and wait about a minute)
seconds = 60 #@param {type:"slider", min:30, max:120, step:10}
try:
    from flaml import AutoML
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "flaml[automl]"], check=True)
    from flaml import AutoML
box("Running", f"AutoML is trying models and settings for {seconds} seconds ...", "step")
Xa_train, Xa_test = prep.fit_transform(X_train), prep.transform(X_test)
automl = AutoML(); automl.fit(Xa_train, y_train, task="classification", metric="accuracy", time_budget=seconds, verbose=0, seed=42)
board = pd.Series({k: 1 - v for k, v in automl.best_loss_per_estimator.items() if v is not None and v < 1}).sort_values(ascending=False)
NAMES = {"lgbm": "LightGBM (gradient boosting)", "xgboost": "XGBoost (gradient boosting)", "xgb_limitdepth": "XGBoost, limited depth", "rf": "Random forest", "extra_tree": "Extra trees", "lrl1": "Logistic regression", "catboost": "CatBoost", "kneighbor": "k-nearest neighbours"}
bars({NAMES.get(k, k): v for k, v in board.items()}, "Best accuracy per model type (validation)", highlight=NAMES.get(board.index[0], board.index[0]), fmt="{:.3f}")
big((NAMES.get(automl.best_estimator, automl.best_estimator), "winner"), (f"{(automl.predict(Xa_test) == y_test).mean():.1%}", "right on the test data"), (f"{pipe.score(X_test, y_test):.1%}", "our own pipeline"))

## Step 4  What AutoML does not check

In [ ]:
#@title Step 4a  Leakage  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "AutoML removes leaking inputs automatically", "AutoML uses every input it gets"]
_ = answer(question, {
    "AutoML removes leaking inputs automatically": (False, "It cannot know when an input becomes available. Give it the final approval date and it will happily use it."),
    "AutoML uses every input it gets": (True, "So the team checks every input first: is it known when the prediction is made?"),
})

In [ ]:
#@title Step 4b  The right metric  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "Accuracy is always the right metric", "It depends on what the service needs"]
_ = answer(question, {
    "Accuracy is always the right metric": (False, "AutoML ranks by the metric it is given. If finding late applications matters most, rank by recall or F1, and a different model may win."),
    "It depends on what the service needs": (True, "AutoML ranks by the metric it is given. The team chooses it, based on what each mistake costs."),
})

In [ ]:
#@title Step 4c  Explainability  (choose, then click &#9654;)
question = "Choose..." #@param ["Choose...", "The winner is always the one to use", "Compare it with a simpler model you can explain"]
_ = answer(question, {
    "The winner is always the one to use": (False, "The winner may be hundreds of trees. If it is barely better than a simple model, the simple one may be the better choice for a public service."),
    "Compare it with a simpler model you can explain": (True, "If the difference is small, choose the model you can explain to applicants and managers. Next exercise: explaining single decisions."),
})

## Result

In [ ]:
#@title Result
box("Do you see it now?", "Better inputs came from knowing the service, not from a smarter algorithm. The pipeline makes sure production sees the data exactly as training did. "
    "AutoML found a strong model in a minute. <b>The team still does the three checks</b>, whether the model comes from AutoML, a vendor or the team itself.", "see")

---
## Stretch (optional, for those who code)

In [ ]:
# Stretch: run AutoML with metric='f1' and see whether another model wins
a2 = AutoML(); a2.fit(Xa_train, y_train, task="classification", metric="f1", time_budget=60, verbose=0, seed=42)
a2.best_estimator